# 코호트 분석 with Python

In [1]:
# import os
# import random #데이터 샘플링
# from collections import Counter # count 용도

import numpy as np
import pandas as pd

# from geopy import distance # 거리 계산
# import geopy.distance
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# 시각화
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
plt.style.use('fivethirtyeight')

# 한글, 마이너스 깨짐 방지
from matplotlib import rc, font_manager, rcParams
font=font_manager.FontProperties(fname="c:/Windows/Fonts/malgun.ttf").get_name()
rc('font', family = font)
rcParams['axes.unicode_minus'] = False

In [3]:
df = pd.read_excel('https://github.com/springcoil/marsmodelling/blob/master/relay-foods.xlsx?raw=true', 
                 sheet_name='Purchase Data - Full Study')
df.head()

,OrderId,OrderDate,UserId,TotalCharges,CommonId,PupId,PickupDate
0,262,2009-01-11,47,50.67,TRQKD,2,2009-01-12
1,278,2009-01-20,47,26.60,4HH2S,3,2009-01-20
2,294,2009-02-03,47,38.71,3TRDC,2,2009-02-04
3,301,2009-02-06,47,53.38,NGAZJ,2,2009-02-09
4,302,2009-02-06,47,14.28,FFYHD,2,2009-02-09


In [4]:
df.shape

(2891, 7)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2891 entries, 0 to 2890
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   OrderId       2891 non-null   int64         
 1   OrderDate     2891 non-null   datetime64[ns]
 2   UserId        2891 non-null   int64         
 3   TotalCharges  2891 non-null   float64       
 4   CommonId      2891 non-null   object        
 5   PupId         2891 non-null   int64         
 6   PickupDate    2891 non-null   datetime64[ns]
dtypes: datetime64[ns](2), float64(1), int64(3), object(1)
memory usage: 158.2+ KB


> 분석 목표

1. 첫 번째 구매행동 이후 몇 개월까지 구매행위가 지속되는가? 
2. 구매주기가 대략적으로 얼만큼 되는가?

+ 첫 구매 날짜(브랜드 인입 시기)에 따른 최근 구매 패턴(구매, 비구매) 비교 등
+ 분석 목표에 따라 변수를 설정해야 한다. 여기에서는 '첫구매'일로 설정!
    - (가입일이 있다면 보통 가입일에 따라서 설정하기도 함)
    - ex. 가입일 후 첫 구매, 재구매, N번째 구매일

## 분석

### 기준이 되는 변수(OrderDate)를 YYYY-MM 형태로 만들어준다.(년월 형식)

In [11]:
df.head()

,OrderId,OrderDate,UserId,TotalCharges,CommonId,PupId,PickupDate
0,262,2009-01-11,47,50.67,TRQKD,2,2009-01-12
1,278,2009-01-20,47,26.60,4HH2S,3,2009-01-20
2,294,2009-02-03,47,38.71,3TRDC,2,2009-02-04
3,301,2009-02-06,47,53.38,NGAZJ,2,2009-02-09
4,302,2009-02-06,47,14.28,FFYHD,2,2009-02-09


In [12]:
df['OrderPeriod'] = df.OrderDate.apply(lambda x: x.strftime('%Y-%m'))
df.head()

,OrderId,OrderDate,UserId,TotalCharges,CommonId,PupId,PickupDate,OrderPeriod
0,262,2009-01-11,47,50.67,TRQKD,2,2009-01-12,2009-01
1,278,2009-01-20,47,26.60,4HH2S,3,2009-01-20,2009-01
2,294,2009-02-03,47,38.71,3TRDC,2,2009-02-04,2009-02
3,301,2009-02-06,47,53.38,NGAZJ,2,2009-02-09,2009-02
4,302,2009-02-06,47,14.28,FFYHD,2,2009-02-09,2009-02


### 각각 사용자의 첫 구매월을 추출하기위해 UserId를 index로 설정한 이후 groupby 함수를 사용하여(기준 index level = 0) 'CohortGroup' 변수를 추가한다.

In [14]:
df.UserId.unique()

array([    47,     95,     98,    112,    141,    147,    160,    177,
          180,    182,    200,    202,    207,    225,    230,    253,
          277,    407,    457,    464,    550,    616,    807,    825,
          891,    899,    937,    940,   1023,   1027,   1409,   1558,
         1588,   1625,   1701,   1844,   1925,   2195,   2235,   2583,
         2930,   3181,   3190,   3194,   3197,   3355,   3385,   3394,
         3616,   3640,   3733,   3767,   3889,   4058,   4152,   4160,
         4244,   4270,   4433,   4516,   4577,   4622,   4640,   4642,
         4658,   4851,   4862,   4903,   4973,   4976,   5378,   5408,
         5423,   5441,   5449,   5534,   5564,   5638,   5654,   5656,
         5657,   5664,   5682,   5686,   5708,   5805,   5831,   5894,
         5977,   5983,   6003,   6012,   6016,   6019,   6069,   6097,
         6153,   6224,   6225,   6245,   6249,   6251,   6261,   6370,
         6379,   6384,   6401,   6408,   6447,   6476,   6578,   6602,
      

In [24]:
df.set_index('UserId').groupby(level=0)['OrderDate'].min() # level 뭐지..?

UserId
47       2009-01-11
95       2009-03-10
98       2009-01-29
112      2009-01-19
141      2009-11-13
            ...    
393616   2010-03-08
394290   2010-03-07
394346   2010-03-07
395039   2010-03-08
396551   2010-03-08
Name: OrderDate, Length: 757, dtype: datetime64[ns]

In [19]:
df.set_index('UserId', inplace=True)

# 고객 각각의 첫 구매기간 추출
df['CohortGroup'] = df.groupby(level=0)['OrderDate'].min().apply(lambda x: x.strftime('%Y-%m'))
df.reset_index(inplace=True)
df.head()

,UserId,OrderId,OrderDate,TotalCharges,CommonId,PupId,PickupDate,OrderPeriod,CohortGroup
0,47,262,2009-01-11,50.67,TRQKD,2,2009-01-12,2009-01,2009-01
1,47,278,2009-01-20,26.60,4HH2S,3,2009-01-20,2009-01,2009-01
2,47,294,2009-02-03,38.71,3TRDC,2,2009-02-04,2009-02,2009-01
3,47,301,2009-02-06,53.38,NGAZJ,2,2009-02-09,2009-02,2009-01
4,47,302,2009-02-06,14.28,FFYHD,2,2009-02-09,2009-02,2009-01


데이터에 오류가 없는지 검증해보기!(항상 데이터를 확인하는 습관을 가지는 것이 좋다.)

In [25]:
df.query('OrderPeriod != CohortGroup')

,UserId,OrderId,OrderDate,TotalCharges,CommonId,PupId,PickupDate,OrderPeriod,CohortGroup
2,47,294,2009-02-03,38.7100,3TRDC,2,2009-02-04,2009-02,2009-01
3,47,301,2009-02-06,53.3800,NGAZJ,2,2009-02-09,2009-02,2009-01
4,47,302,2009-02-06,14.2800,FFYHD,2,2009-02-09,2009-02,2009-01
5,47,321,2009-02-17,29.5000,HA5R3,3,2009-02-17,2009-02,2009-01
6,47,333,2009-02-23,18.9100,RSXQG,2,2009-02-23,2009-02,2009-01
...,...,...,...,...,...,...,...,...,...
2847,369133,3094,2010-03-01,17.7900,SAPNJ,7,2010-03-04,2010-03,2010-02
2849,370402,3119,2010-03-02,37.3552,ZLBJ5,9,2010-03-03,2010-03,2010-02
2862,376775,3227,2010-03-08,43.7000,Q53K9,9,2010-03-10,2010-03,2010-02
2868,380119,3224,2010-03-08,56.7115,ZRWZB,15,2010-03-08,2010-03,2010-02


In [27]:
df.query('UserId==47')

,UserId,OrderId,OrderDate,TotalCharges,CommonId,PupId,PickupDate,OrderPeriod,CohortGroup
0,47,262,2009-01-11,50.6700,TRQKD,2,2009-01-12,2009-01,2009-01
1,47,278,2009-01-20,26.6000,4HH2S,3,2009-01-20,2009-01,2009-01
2,47,294,2009-02-03,38.7100,3TRDC,2,2009-02-04,2009-02,2009-01
3,47,301,2009-02-06,53.3800,NGAZJ,2,2009-02-09,2009-02,2009-01
4,47,302,2009-02-06,14.2800,FFYHD,2,2009-02-09,2009-02,2009-01
...,...,...,...,...,...,...,...,...,...
71,47,2997,2010-02-23,27.9700,ZCSXX,15,2010-02-26,2010-02,2009-01
72,47,3037,2010-02-25,15.5025,HG86F,15,2010-02-26,2010-02,2009-01
73,47,3082,2010-03-01,48.1520,P98FG,18,2010-03-01,2010-03,2009-01
74,47,3109,2010-03-02,34.1300,4PFP8,3,2010-03-02,2010-03,2009-01


### 첫 구매일(년월)과 구매 날짜(년월)를 기준으로 하여 고객 수, 주문 수, 총 매출 합계를 계산한다.

In [28]:
# CohortGroup & OrderPeriod
grouped = df.groupby(['CohortGroup', 'OrderPeriod'])

cohorts = grouped.agg({'UserId' : pd.Series.nunique, # DISTINCT COUNT
                       'OrderId' : pd.Series.nunique,
                       'TotalCharges' : np.sum}) # SUM

cohorts.rename(columns={'UserId' : 'TotalUsers',
                        'OrderId' : 'TotalOrders'}, inplace=True)

cohorts.head()

TotalUsers  TotalOrders  TotalCharges
CohortGroup OrderPeriod                                       
2009-01     2009-01              22           30      1850.255
            2009-02               8           25      1351.065
            2009-03              10           26      1357.360
            2009-04               9           28      1604.500
            2009-05              10           26      1575.625

### <년월 - 년월>의 패턴을 <년월 - 소요기간(월)>로 변환하여 보기위해 다음과 같은 함수를 만들어준다.

In [30]:
# Label the CohortPeriod for each CohortGroup
def cohort_period(df):
    """
    Creates a `CohortPeriod` column, which is the Nth period based on the user's first purchase.
    
    Example
    -------
    Say you want to get the 3rd month for every user:
        df.sort(['UserId', 'OrderTime', inplace=True)
        df = df.groupby('UserId').apply(cohort_period)
        df[df.CohortPeriod == 3]
    """
    df['CohortPeriod'] = np.arange(len(df)) + 1
    return df

In [32]:
cohorts = cohorts.groupby(level=0).apply(cohort_period)
cohorts

TotalUsers  TotalOrders  TotalCharges  CohortPeriod
CohortGroup OrderPeriod                                                     
2009-01     2009-01              22           30     1850.2550             1
            2009-02               8           25     1351.0650             2
            2009-03              10           26     1357.3600             3
            2009-04               9           28     1604.5000             4
            2009-05              10           26     1575.6250             5
...                             ...          ...           ...           ...
2010-01     2010-02              50          101     8453.1039             2
            2010-03              26           31     2238.6461             3
2010-02     2010-02             100          139     7374.7108             1
            2010-03              19           19      945.9633             2
2010-03     2010-03              24           26     1099.5471             1

[119 rows x 4 columns]

### 데이터가 맞게 정리되었는지 확인하기

**assert 함수 : 괄호 안의 결과가 False라면 예외(exception)가 발생한다.**

In [49]:
x = df[(df.CohortGroup=='2009-01') & (df.OrderPeriod=='2009-01')]
# y = cohorts.ix[('2009-01', '2009-01')] # ix 누구세요?
y = cohorts.loc[('2009-01', '2009-01')] # ix는 구식! 없어짐

# 가정 설정문 / 이는 단순히 에러를 찾는것이 아니라 값을 보증하기 위해 사용된다.
assert(x['UserId'].nunique() == y['TotalUsers'])
assert(x['TotalCharges'].sum().round(2) == y['TotalCharges'].round(2))
assert(x['OrderId'].nunique() == y['TotalOrders'])

In [47]:
cohorts.loc[('2009-01', '2009-01')]

TotalUsers        22.000
TotalOrders       30.000
TotalCharges    1850.255
CohortPeriod       1.000
Name: (2009-01, 2009-01), dtype: float64

In [50]:
x = df[(df.CohortGroup=='2009-05') & (df.OrderPeriod=='2009-09')]
y = cohorts.loc[('2009-05', '2009-09')]

assert(x['UserId'].nunique() == y['TotalUsers'])
assert(x['TotalCharges'].sum().round(2) == y['TotalCharges'].round(2))
assert(x['OrderId'].nunique() == y['TotalOrders'])

### Retention 결과를 (%) 비율로 나타내기 위해 각각 첫 구매일(년월)에 따른 회원수를 구한다.

In [52]:
cohorts.reset_index()

,CohortGroup,OrderPeriod,TotalUsers,TotalOrders,TotalCharges,CohortPeriod
0,2009-01,2009-01,22,30,1850.2550,1
1,2009-01,2009-02,8,25,1351.0650,2
2,2009-01,2009-03,10,26,1357.3600,3
3,2009-01,2009-04,9,28,1604.5000,4
4,2009-01,2009-05,10,26,1575.6250,5
...,...,...,...,...,...,...
114,2010-01,2010-02,50,101,8453.1039,2
115,2010-01,2010-03,26,31,2238.6461,3
116,2010-02,2010-02,100,139,7374.7108,1
117,2010-02,2010-03,19,19,945.9633,2
